# 03 — Test-set evaluation, error analysis and prediction on new text
The saved model is loaded from `model/` (no retraining).

In [ ]:
import os, sys
# Colab: clone the repository first
if "google.colab" in sys.modules and not os.path.exists("src"):
    !git clone https://github.com/<username>/kz-profanity-detection-DL26.git
    %cd kz-profanity-detection-DL26
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Working dir:", os.getcwd())

In [ ]:
import json
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
from src.predict import load_model, predict
from src.preprocess import normalize

bundle = load_model("model")
print("Threshold:", bundle["cfg"]["threshold"])

## 1. Test-set prediction (run once)

In [ ]:
test = pd.read_csv("data/test.csv")
preds = [predict(bundle, t) for t in test.text]
test["prob"] = [p["probability"] for p in preds]
test["pred"] = [p["label"] for p in preds]

P, R, F, _ = precision_recall_fscore_support(test.label, test.pred, average="binary")
cm = confusion_matrix(test.label, test.pred)
tn, fp, fn, tp = cm.ravel()
metrics = {"n_test": len(test), "n_clean": int((test.label == 0).sum()), "n_profane": int(test.label.sum()),
           "precision": round(P, 4), "recall": round(R, 4), "f1": round(F, 4),
           "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
           "threshold": bundle["cfg"]["threshold"]}
json.dump(metrics, open("results/test_metrics.json", "w"), indent=2)
metrics

In [ ]:
fig, ax = plt.subplots(figsize=(4.2, 3.6))
ax.imshow(cm, cmap="Blues")
names = ["clean (0)", "profane (1)"]
ax.set_xticks([0, 1], names); ax.set_yticks([0, 1], names)
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=13,
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_title(f"Test set (n={len(test)})")
plt.tight_layout(); plt.savefig("results/confusion_matrix.png", dpi=150); plt.show()

**False positive (FP):** a clean review flagged as profane → it may be hidden unfairly.
**False negative (FN):** a profane review the model missed → users still see it.

## 2. Errors and hard examples

In [ ]:
test[["text", "label", "pred", "prob", "domain", "matched"]].to_csv("results/test_predictions.csv", index=False)
pd.set_option("display.max_colwidth", 120)
print("False positives:")
display(test[(test.label == 0) & (test.pred == 1)][["text", "prob", "domain"]])
print("False negatives:")
display(test[(test.label == 1) & (test.pred == 0)][["text", "prob", "matched"]])

In [ ]:
# Borderline cases: probability close to the threshold
t = bundle["cfg"]["threshold"]
test.assign(dist=(test.prob - t).abs()).sort_values("dist").head(10)[["text", "label", "pred", "prob"]]

## 3. Prediction on new comments
Includes spellings that are **not** in the keyword list, to check generalization beyond it.

In [ ]:
examples = [
    "Сапасы өте нашар, ақшамды қайтарыңыз",          # negative but clean
    "Жеткізу кешікті, курьер ақымақ екен",            # mild insult -> 0 by our rules
    "Керемет тауар, рахмет!",
    "kotakbas magazin",                                # Latin script
    "қ0тақ тауар екен",                                # digit instead of letter
    "с*к* сапасы жоқ",                                 # masked
]
for e in examples:
    print(predict(bundle, e), "|", e)

In [ ]:
# Custom text
predict(bundle, "Мәтінді осында жазыңыз")